# Лабораторная работа 1: Бинарное решающее дерево ID3 с критерием Джини

## 1. Загрузка и подготовка датасета

In [2]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from source import (
    load_adult_dataset,
    introduce_missing_values,
    get_feature_info,
    Binarizer,
    ID3Tree,
    reduced_error_pruning,
)

In [3]:
# Датасет Adult: категориальные и количественные признаки, пропуски в workclass и occupation
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
df = load_adult_dataset(url)
print(f"Размер: {df.shape}")
print(f"Пропуски:\n{df.isna().sum()[df.isna().sum() > 0]}")
cat_cols, num_cols = get_feature_info(df)
print(f"\nКатегориальные: {cat_cols}")
print(f"Количественные: {num_cols}")

Размер: (32561, 15)
Пропуски:
workclass         1836
occupation        1843
native_country     583
dtype: int64

Категориальные: ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']
Количественные: ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']


## 2. Бинаризация признаков и разбиение на выборки

In [5]:
X_raw = df.drop(columns=["income"])
y = df["income"].values
binarizer = Binarizer()
X = binarizer.fit_transform(X_raw, cat_cols, num_cols)
print(f"После бинаризации: {X.shape[1]} признаков")

После бинаризации: 105 признаков


In [6]:
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42)

## 3. Обучение дерева ID3 с критерием Джини

In [7]:
tree = ID3Tree(max_depth=12, min_samples_leaf=10, holdout_frac=0.2, random_state=42)
tree.fit(X_train, y_train)
print(f"Глубина: {tree.get_depth()}, листьев: {tree.count_leaves()}")

Глубина: 12, листьев: 415


## 4. Оценка качества до редукции

In [8]:
acc_train_before = accuracy_score(y_train, tree.predict(X_train))
acc_val_before = accuracy_score(y_val, tree.predict(X_val))
acc_test_before = accuracy_score(y_test, tree.predict(X_test))
print(f"До редукции: train={acc_train_before:.4f}, val={acc_val_before:.4f}, test={acc_test_before:.4f}")

До редукции: train=0.8320, val=0.8170, test=0.8162


## 5. Редукция дерева

In [9]:
reduced_error_pruning(tree, X_val, y_val)
print(f"После редукции: глубина={tree.get_depth()}, листьев={tree.count_leaves()}")

После редукции: глубина=12, листьев=85


## 6. Оценка качества после редукции и сравнение

In [10]:
acc_train_after = accuracy_score(y_train, tree.predict(X_train))
acc_val_after = accuracy_score(y_val, tree.predict(X_val))
acc_test_after = accuracy_score(y_test, tree.predict(X_test))
print(f"После редукции: train={acc_train_after:.4f}, val={acc_val_after:.4f}, test={acc_test_after:.4f}")

После редукции: train=0.8386, val=0.8342, test=0.8276


In [11]:
print("Сравнение ID3 (после редукции) vs sklearn DecisionTreeClassifier (с критерием Gini):")
# sklearn не поддерживает NaN — заполняем 0 для сравнения
X_train_fill = X_train.fillna(0)
X_test_fill = X_test.fillna(0)
sk_tree = DecisionTreeClassifier(criterion="gini", max_depth=12, min_samples_leaf=10, random_state=42)
sk_tree.fit(X_train_fill, y_train)
sk_acc = accuracy_score(y_test, sk_tree.predict(X_test_fill))
print(f"  ID3 (с пропусками, редукция): test={acc_test_after:.4f}")
print(f"  sklearn DecisionTree:         test={sk_acc:.4f}")

Сравнение ID3 (после редукции) vs sklearn DecisionTreeClassifier (с критерием Gini):
  ID3 (с пропусками, редукция): test=0.8276
  sklearn DecisionTree:         test=0.8311


In [12]:
print(classification_report(y_test, tree.predict(X_test), target_names=["<=50K", ">50K"]))
print(confusion_matrix(y_test, tree.predict(X_test)))

              precision    recall  f1-score   support

       <=50K       0.86      0.93      0.89      4980
        >50K       0.68      0.50      0.58      1533

    accuracy                           0.83      6513
   macro avg       0.77      0.71      0.73      6513
weighted avg       0.82      0.83      0.82      6513

[[4622  358]
 [ 765  768]]


In [13]:
# Сводная таблица
pd.DataFrame({
    "": ["До редукции", "После редукции"],
    "train": [acc_train_before, acc_train_after],
    "val": [acc_val_before, acc_val_after],
    "test": [acc_test_before, acc_test_after],
}).set_index("")

,train,val,test
,,,
До редукции,0.831951,0.816953,0.816214
После редукции,0.838554,0.834152,0.827576
